In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
movie_cast_df = (
    spark.read
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/movie_cast.json")
    )
display(movie_cast_df)

castOrder,characterName,genderId,movieId,personId
0,Captain Jack Sparrow,2,285,85
1,Will Turner,2,285,114
2,Elizabeth Swann,1,285,116
3,William Bootstrap Bill Turner,2,285,1640
4,Captain Sao Feng,2,285,1619
5,Captain Davy Jones,2,285,2440
6,Captain Hector Barbossa,2,285,118
7,Admiral James Norrington,2,285,1709
8,Joshamee Gibbs,2,285,2449
9,Lord Cutler Beckett,2,285,2441


In [0]:
movie_cast_df.columns

['castOrder', 'characterName', 'genderId', 'movieId', 'personId']

In [0]:
movie_cast_schema = StructType([
    StructField("castOrder", IntegerType(), True),
    StructField("characterName", StringType(), True),
    StructField("genderId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("personId", IntegerType(), True)
])
movie_cast_df = (
    spark.read
        .schema(movie_cast_schema)
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/movie_cast.json")
)

In [0]:
movie_cast_df = (
    add_ingestion_date(movie_cast_df)
        .select(
            col("movieId").alias("movie_id"),
            col("personId").alias("person_id"),
            col("characterName").alias("character_name"),
            lit(v_env).alias("env")
        )
)
display(movie_cast_df)

movie_id,person_id,character_name,env
285,85,Captain Jack Sparrow,dev
285,114,Will Turner,dev
285,116,Elizabeth Swann,dev
285,1640,William Bootstrap Bill Turner,dev
285,1619,Captain Sao Feng,dev
285,2440,Captain Davy Jones,dev
285,118,Captain Hector Barbossa,dev
285,1709,Admiral James Norrington,dev
285,2449,Joshamee Gibbs,dev
285,2441,Lord Cutler Beckett,dev


In [0]:
(
    movie_cast_df
        .write
        .format("parquet")
        .mode("overwrite")
        .save(f"{silver_folder_path}/movie_cast")
)

In [0]:
dbutils.notebook.exit("Success")